# Сравнение различных видов рекуррентных сетей

## Часть 1: Настройка окружения

Инициализация зависимостей, логирования и параметров визуализации.

In [1]:
import logging
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from collections import defaultdict

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, TensorDataset
from sklearn.preprocessing import StandardScaler

from sklearn.metrics import (
    roc_auc_score, precision_recall_curve, roc_curve,
    f1_score, fbeta_score, precision_score, recall_score,
    accuracy_score, confusion_matrix, classification_report,
    average_precision_score
)

import optuna
from optuna.pruners import MedianPruner
from optuna.samplers import TPESampler

# Setup
warnings.filterwarnings('ignore')
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')
logger = logging.getLogger(__name__)

# PyTorch settings
if torch.cuda.is_available():
    device = torch.device('cuda')
elif torch.backends.mps.is_available():
    device = torch.device('mps')
else:
    device = torch.device('cpu')
torch.manual_seed(42)
np.random.seed(42)

# Visualization settings
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")
logger.info(f"Окружение инициализировано. Устройство: {device}")

/Users/berdov/project3/fraud_detection/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
2026-04-05 16:17:59,419 - INFO - Окружение инициализировано. Устройство: mps


## Часть 2: Загрузка и подготовка данных

Загрузка транзакций и организация их в последовательности по клиентам.

In [2]:
import random

def reset_seeds(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)

In [3]:
df_train = pd.read_csv("train.csv")
df_val = pd.read_csv("val.csv")
df_test = pd.read_csv("test.csv")

logger.info(f"Тренировочный набор: {df_train.shape[0]:,} транзакций x {df_train.shape[1]} признаков")
logger.info(f"Валидационный набор: {df_val.shape[0]:,} транзакций x {df_val.shape[1]} признаков")
logger.info(f"Тестовый набор: {df_test.shape[0]:,} транзакций x {df_test.shape[1]} признаков")
logger.info(f"Доля мошенничества (тренировка): {df_train['FraudResult'].mean():.2%}")
logger.info(f"Доля мошенничества (валидация): {df_val['FraudResult'].mean():.2%}")
logger.info(f"Доля мошенничества (тест): {df_test['FraudResult'].mean():.2%}")

print("\nСводка данных:")
print(f"  Уникальных клиентов (тренировка): {df_train['CustomerId'].nunique():,}")
print(f"  Уникальных клиентов (валидация): {df_val['CustomerId'].nunique():,}")
print(f"  Уникальных клиентов (тест): {df_test['CustomerId'].nunique():,}")
print(f"  Средний размер последовательности: {len(df_train) / df_train['CustomerId'].nunique():.1f} транзакций")

2026-04-05 16:17:59,529 - INFO - Тренировочный набор: 54,459 транзакций x 10 признаков
2026-04-05 16:17:59,529 - INFO - Валидационный набор: 21,537 транзакций x 10 признаков
2026-04-05 16:17:59,529 - INFO - Тестовый набор: 19,666 транзакций x 10 признаков
2026-04-05 16:17:59,531 - INFO - Доля мошенничества (тренировка): 0.17%
2026-04-05 16:17:59,531 - INFO - Доля мошенничества (валидация): 0.21%
2026-04-05 16:17:59,531 - INFO - Доля мошенничества (тест): 0.29%



Сводка данных:
  Уникальных клиентов (тренировка): 2,245
  Уникальных клиентов (валидация): 748
  Уникальных клиентов (тест): 749
  Средний размер последовательности: 24.3 транзакций


## Часть 3: Инженерия признаков и подготовка последовательностей

Извлечение признаков и организация данных в последовательности для обработки GRU.

In [4]:
def prepare_transaction_features(df):
    """
    Подготовка признаков для каждой транзакции.
    """
    df = df.copy()
    
    id_columns = ["ProviderId", "ProductId", "ChannelId"]
    
    for col in id_columns:
        if col in df.columns:
            df[col] = df[col].astype(str).str.extract(r"(\d+)$", expand=False).astype("Int64")
    
    if "TransactionStartTime" in df.columns:
        df["TransactionStartTime"] = pd.to_datetime(df["TransactionStartTime"], utc=True)
        df["tx_hour"] = df["TransactionStartTime"].dt.hour
        df["tx_dayofweek"] = df["TransactionStartTime"].dt.dayofweek
        df["tx_month"] = df["TransactionStartTime"].dt.month
        df["tx_is_weekend"] = (df["TransactionStartTime"].dt.dayofweek >= 5).astype(int)
        df = df.drop(columns=["TransactionStartTime"])
    
    return df


logger.info("Подготовка признаков для транзакций...")
train_features = prepare_transaction_features(df_train)
val_features = prepare_transaction_features(df_val)
test_features = prepare_transaction_features(df_test)

# Единое кодирование ProductCategory по train
if "ProductCategory" in train_features.columns:
    train_features["ProductCategory"] = train_features["ProductCategory"].fillna("__MISSING__").astype(str)
    val_features["ProductCategory"] = val_features["ProductCategory"].fillna("__MISSING__").astype(str)
    test_features["ProductCategory"] = test_features["ProductCategory"].fillna("__MISSING__").astype(str)

    product_category_mapping = {
        category: idx
        for idx, category in enumerate(train_features["ProductCategory"].unique())
    }

    train_features["ProductCategory"] = (
        train_features["ProductCategory"]
        .map(product_category_mapping)
        .fillna(-1)
        .astype(int)
    )

    val_features["ProductCategory"] = (
        val_features["ProductCategory"]
        .map(product_category_mapping)
        .fillna(-1)
        .astype(int)
    )

    test_features["ProductCategory"] = (
        test_features["ProductCategory"]
        .map(product_category_mapping)
        .fillna(-1)
        .astype(int)
    )

# Выравнивание колонок во всех датасетах
common_cols = list(set(train_features.columns) & set(val_features.columns) & set(test_features.columns))
common_cols = sorted(common_cols)

train_features = train_features[common_cols]
val_features = val_features[common_cols]
test_features = test_features[common_cols]

logger.info(f"Признаки подготовлены - {train_features.shape[1]} признаков")
logger.info(f"Выровненные колонки: {common_cols}")

2026-04-05 16:17:59,543 - INFO - Подготовка признаков для транзакций...
2026-04-05 16:17:59,700 - INFO - Признаки подготовлены - 13 признаков
2026-04-05 16:17:59,701 - INFO - Выровненные колонки: ['Amount', 'ChannelId', 'CustomerId', 'FraudResult', 'PricingStrategy', 'ProductCategory', 'ProductId', 'ProviderId', 'TransactionId', 'tx_dayofweek', 'tx_hour', 'tx_is_weekend', 'tx_month']


### Создание последовательностей для каждого клиента

In [5]:
def create_client_sequences(df, max_seq_len=100):
    """
    Организация транзакций в последовательности по клиентам.
    
    Args:
        df: DataFrame с транзакциями
        max_seq_len: максимальная длина последовательности
        
    Returns:
        sequences: список массивов признаков
        labels: список меток (fraud/not fraud)
        client_ids: список ID клиентов
    """
    # Определение признаков (исключаем ID и целевую переменную)
    TARGET_COL = "FraudResult"
    CLIENT_ID_COL = "CustomerId"
    
    exclude_cols = {
        TARGET_COL, "TransactionId", "BatchId", "SubscriptionId",
        "CustomerId", "quantile"
    }
    
    feature_cols = [c for c in df.columns if c not in exclude_cols]
    
    # Группировка по клиентам
    sequences = []
    labels = []
    client_ids = []
    
    for account_id, group in df.groupby(CLIENT_ID_COL, sort=False):
        # Извлечение признаков для последовательности
        seq = group[feature_cols].values.astype(np.float32)
        
        # Обрезка или дополнение
        if len(seq) > max_seq_len:
            seq = seq[-max_seq_len:]  # берем последние транзакции
        
        # Padding нулями в начало
        if len(seq) < max_seq_len:
            padding = np.zeros((max_seq_len - len(seq), seq.shape[1]), dtype=np.float32)
            seq = np.vstack([padding, seq])
        
        # Целевая переменная: был ли хотя бы один fraud у клиента
        label = int(group[TARGET_COL].max())
        
        sequences.append(seq)
        labels.append(label)
        client_ids.append(account_id)
    
    return np.array(sequences), np.array(labels), np.array(client_ids), feature_cols


logger.info("Создание последовательностей клиентов...")
train_sequences, train_labels, train_clients, feature_cols = create_client_sequences(train_features)
val_sequences, val_labels, val_clients, _ = create_client_sequences(val_features)
test_sequences, test_labels, test_clients, _ = create_client_sequences(test_features)

logger.info(f"Тренировочные последовательности: {train_sequences.shape}")
logger.info(f"Валидационные последовательности: {val_sequences.shape}")
logger.info(f"Тестовые последовательности: {test_sequences.shape}")
logger.info(f"Распределение меток (тренировка): {np.bincount(train_labels)}")
logger.info(f"Распределение меток (валидация): {np.bincount(val_labels)}")
logger.info(f"Распределение меток (тест): {np.bincount(test_labels)}")
logger.info(f"Всего признаков в последовательности: {len(feature_cols)}")

print(f"\nРазмер последовательностей:")
print(f"  Максимальная длина: 100 транзакций")
print(f"  Размер признаков: {train_sequences.shape[2]}")
print(f"  Клиентов (тренировка): {len(train_sequences)}")
print(f"  Клиентов (валидация): {len(val_sequences)}")
print(f"  Клиентов (тест): {len(test_sequences)}")

2026-04-05 16:17:59,711 - INFO - Создание последовательностей клиентов...
2026-04-05 16:18:00,488 - INFO - Тренировочные последовательности: (2245, 100, 10)
2026-04-05 16:18:00,489 - INFO - Валидационные последовательности: (748, 100, 10)
2026-04-05 16:18:00,489 - INFO - Тестовые последовательности: (749, 100, 10)
2026-04-05 16:18:00,489 - INFO - Распределение меток (тренировка): [2213   32]
2026-04-05 16:18:00,489 - INFO - Распределение меток (валидация): [735  13]
2026-04-05 16:18:00,490 - INFO - Распределение меток (тест): [740   9]
2026-04-05 16:18:00,490 - INFO - Всего признаков в последовательности: 10



Размер последовательностей:
  Максимальная длина: 100 транзакций
  Размер признаков: 10
  Клиентов (тренировка): 2245
  Клиентов (валидация): 748
  Клиентов (тест): 749


### Нормализация признаков

In [6]:
# Нормализация последовательностей
# Reshape для StandardScaler: (n_clients, seq_len, n_features) -> (n_clients*seq_len, n_features)
n_clients, seq_len, n_features = train_sequences.shape
training_reshaped = train_sequences.reshape(-1, n_features)

scaler = StandardScaler()
training_scaled_reshaped = scaler.fit_transform(training_reshaped)
training_sequences_scaled = training_scaled_reshaped.reshape(n_clients, seq_len, n_features)

# Применение того же scaler к валидации
val_reshaped = val_sequences.reshape(-1, n_features)
val_scaled_reshaped = scaler.transform(val_reshaped)
val_sequences_scaled = val_scaled_reshaped.reshape(val_sequences.shape[0], seq_len, n_features)

# Применение того же scaler к тесту
test_reshaped = test_sequences.reshape(-1, n_features)
test_scaled_reshaped = scaler.transform(test_reshaped)
test_sequences_scaled = test_scaled_reshaped.reshape(test_sequences.shape[0], seq_len, n_features)

logger.info("Признаки нормализованы")
print(f"Нормализация - Статистика тренировки:")
print(f"  Mean: {training_sequences_scaled.reshape(-1, n_features).mean(axis=0)[:5]}...")
print(f"  Std:  {training_sequences_scaled.reshape(-1, n_features).std(axis=0)[:5]}...")

2026-04-05 16:18:00,518 - INFO - Признаки нормализованы


Нормализация - Статистика тренировки:
  Mean: [ 1.8011113e-05 -1.5995996e-04  2.7012502e-05  1.8363736e-04
  2.7989814e-04]...
  Std:  [0.9996074  0.99843687 1.0012281  0.99954414 1.0011429 ]...


### Преобразование в PyTorch tensors

In [7]:
# Преобразование в PyTorch
X_train_tensor = torch.FloatTensor(training_sequences_scaled)
y_train_tensor = torch.LongTensor(train_labels)

X_val_tensor = torch.FloatTensor(val_sequences_scaled)
y_val_tensor = torch.LongTensor(val_labels)

X_test_tensor = torch.FloatTensor(test_sequences_scaled)
y_test_tensor = torch.LongTensor(test_labels)

logger.info(f"Тренировочные тензоры: {X_train_tensor.shape}")
logger.info(f"Валидационные тензоры: {X_val_tensor.shape}")
logger.info(f"Тестовые тензоры: {X_test_tensor.shape}")

2026-04-05 16:18:00,530 - INFO - Тренировочные тензоры: torch.Size([2245, 100, 10])
2026-04-05 16:18:00,530 - INFO - Валидационные тензоры: torch.Size([748, 100, 10])
2026-04-05 16:18:00,530 - INFO - Тестовые тензоры: torch.Size([749, 100, 10])


## Часть 4: Архитектуры рекуррентных моделей для последовательностей

In [8]:
class ImprovedGRUClassifier(nn.Module):
    """
    Улучшенный классификатор на основе GRU для последовательностей транзакций.
    Идея:
    - проекция входа
    - bidirectional GRU
    - LayerNorm
    - attention pooling
    - mean pooling
    - max pooling
    - усиленная classifier head
    """
    def __init__(
        self,
        input_dim,
        hidden_dim=64,
        num_layers=2,
        num_classes=2,
        dropout=0.3,
        bidirectional=True
    ):
        super().__init__()
        
        self.hidden_dim = hidden_dim
        self.num_layers = num_layers
        self.bidirectional = bidirectional
        self.num_directions = 2 if bidirectional else 1
        
        # Входная проекция
        self.input_proj = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout)
        )
        
        # GRU
        self.gru = nn.GRU(
            input_size=hidden_dim,
            hidden_size=hidden_dim,
            num_layers=num_layers,
            dropout=dropout if num_layers > 1 else 0.0,
            batch_first=True,
            bidirectional=bidirectional
        )
        
        gru_out_dim = hidden_dim * self.num_directions
        
        # Нормализация выходов RNN
        self.layer_norm = nn.LayerNorm(gru_out_dim)
        
        # Attention scoring
        self.attention = nn.Sequential(
            nn.Linear(gru_out_dim, gru_out_dim // 2),
            nn.Tanh(),
            nn.Linear(gru_out_dim // 2, 1)
        )
        
        self.dropout = nn.Dropout(dropout)
        
        # Будем объединять:
        # 1) attention pooled
        # 2) mean pooled
        # 3) max pooled
        # 4) last hidden
        combined_dim = gru_out_dim * 4
        
        self.classifier = nn.Sequential(
            nn.Linear(combined_dim, 64),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(64, 32),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(32, num_classes)
        )
    
    def forward(self, x):
        """
        x: (batch, seq_len, input_dim)
        """
        # 1. Проекция входа
        x = self.input_proj(x)   # (batch, seq_len, hidden_dim)
        
        # 2. GRU
        gru_out, h_n = self.gru(x)   # (batch, seq_len, hidden_dim * directions)
        gru_out = self.layer_norm(gru_out)
        
        # 3. Последний hidden state
        if self.bidirectional:
            # h_n shape: (num_layers * 2, batch, hidden_dim)
            last_forward = h_n[-2]
            last_backward = h_n[-1]
            last_hidden = torch.cat([last_forward, last_backward], dim=1)
        else:
            last_hidden = h_n[-1]
        
        # 4. Attention pooling
        attn_scores = self.attention(gru_out)              # (batch, seq_len, 1)
        attn_weights = torch.softmax(attn_scores, dim=1)   # (batch, seq_len, 1)
        attn_pooled = torch.sum(gru_out * attn_weights, dim=1)  # (batch, gru_out_dim)
        
        # 5. Mean pooling
        mean_pooled = torch.mean(gru_out, dim=1)
        
        # 6. Max pooling
        max_pooled, _ = torch.max(gru_out, dim=1)
        
        # 7. Объединение признаков
        features = torch.cat(
            [last_hidden, attn_pooled, mean_pooled, max_pooled],
            dim=1
        )
        
        features = self.dropout(features)
        
        # 8. Классификация
        logits = self.classifier(features)
        return logits

class RNNClassifier(nn.Module):
    """
    Базовый классификатор на основе простой RNN.
    """
    def __init__(
        self,
        input_dim,
        hidden_dim=32,
        num_layers=1,
        num_classes=2,
        dropout=0.2,
        bidirectional=False
    ):
        super().__init__()

        self.hidden_dim = hidden_dim
        self.num_layers = num_layers
        self.bidirectional = bidirectional
        self.num_directions = 2 if bidirectional else 1

        self.input_proj = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout)
        )

        self.rnn = nn.RNN(
            input_size=hidden_dim,
            hidden_size=hidden_dim,
            num_layers=num_layers,
            nonlinearity='tanh',
            dropout=dropout if num_layers > 1 else 0.0,
            batch_first=True,
            bidirectional=bidirectional
        )

        output_dim = hidden_dim * self.num_directions

        self.dropout = nn.Dropout(dropout)
        self.classifier = nn.Sequential(
            nn.Linear(output_dim, 32),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(32, num_classes)
        )

    def forward(self, x):
        x = self.input_proj(x)
        rnn_out, h_n = self.rnn(x)

        if self.bidirectional:
            last_forward = h_n[-2]
            last_backward = h_n[-1]
            last_hidden = torch.cat([last_forward, last_backward], dim=1)
        else:
            last_hidden = h_n[-1]

        last_hidden = self.dropout(last_hidden)
        logits = self.classifier(last_hidden)
        return logits


class LSTMClassifier(nn.Module):
    """
    Базовый классификатор на основе LSTM.
    """
    def __init__(
        self,
        input_dim,
        hidden_dim=32,
        num_layers=2,
        num_classes=2,
        dropout=0.3,
        bidirectional=False
    ):
        super().__init__()

        self.hidden_dim = hidden_dim
        self.num_layers = num_layers
        self.bidirectional = bidirectional
        self.num_directions = 2 if bidirectional else 1

        self.input_proj = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout)
        )

        self.lstm = nn.LSTM(
            input_size=hidden_dim,
            hidden_size=hidden_dim,
            num_layers=num_layers,
            dropout=dropout if num_layers > 1 else 0.0,
            batch_first=True,
            bidirectional=bidirectional
        )

        output_dim = hidden_dim * self.num_directions

        self.dropout = nn.Dropout(dropout)
        self.classifier = nn.Sequential(
            nn.Linear(output_dim, 32),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(32, num_classes)
        )

    def forward(self, x):
        x = self.input_proj(x)
        lstm_out, (h_n, c_n) = self.lstm(x)

        if self.bidirectional:
            last_forward = h_n[-2]
            last_backward = h_n[-1]
            last_hidden = torch.cat([last_forward, last_backward], dim=1)
        else:
            last_hidden = h_n[-1]

        last_hidden = self.dropout(last_hidden)
        logits = self.classifier(last_hidden)
        return logits


class GRUBaseClassifier(nn.Module):
    """
    Базовый классификатор на основе обычной GRU.
    """
    def __init__(
        self,
        input_dim,
        hidden_dim=32,
        num_layers=2,
        num_classes=2,
        dropout=0.3,
        bidirectional=False
    ):
        super().__init__()

        self.hidden_dim = hidden_dim
        self.num_layers = num_layers
        self.bidirectional = bidirectional
        self.num_directions = 2 if bidirectional else 1

        self.input_proj = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout)
        )

        self.gru = nn.GRU(
            input_size=hidden_dim,
            hidden_size=hidden_dim,
            num_layers=num_layers,
            dropout=dropout if num_layers > 1 else 0.0,
            batch_first=True,
            bidirectional=bidirectional
        )

        output_dim = hidden_dim * self.num_directions

        self.dropout = nn.Dropout(dropout)
        self.classifier = nn.Sequential(
            nn.Linear(output_dim, 32),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(32, num_classes)
        )

    def forward(self, x):
        x = self.input_proj(x)
        gru_out, h_n = self.gru(x)

        if self.bidirectional:
            last_forward = h_n[-2]
            last_backward = h_n[-1]
            last_hidden = torch.cat([last_forward, last_backward], dim=1)
        else:
            last_hidden = h_n[-1]

        last_hidden = self.dropout(last_hidden)
        logits = self.classifier(last_hidden)
        return logits
    
class ImprovedLSTMClassifier(nn.Module):
    """
    Улучшенный классификатор на основе LSTM для последовательностей транзакций.
    Идея:
    - проекция входа
    - bidirectional LSTM
    - LayerNorm
    - attention pooling
    - mean pooling
    - max pooling
    - усиленная classifier head
    """
    def __init__(
        self,
        input_dim,
        hidden_dim=64,
        num_layers=2,
        num_classes=2,
        dropout=0.3,
        bidirectional=True
    ):
        super().__init__()

        self.hidden_dim = hidden_dim
        self.num_layers = num_layers
        self.bidirectional = bidirectional
        self.num_directions = 2 if bidirectional else 1

        self.input_proj = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout)
        )

        self.lstm = nn.LSTM(
            input_size=hidden_dim,
            hidden_size=hidden_dim,
            num_layers=num_layers,
            dropout=dropout if num_layers > 1 else 0.0,
            batch_first=True,
            bidirectional=bidirectional
        )

        lstm_out_dim = hidden_dim * self.num_directions

        self.layer_norm = nn.LayerNorm(lstm_out_dim)

        self.attention = nn.Sequential(
            nn.Linear(lstm_out_dim, lstm_out_dim // 2),
            nn.Tanh(),
            nn.Linear(lstm_out_dim // 2, 1)
        )

        self.dropout = nn.Dropout(dropout)

        combined_dim = lstm_out_dim * 4

        self.classifier = nn.Sequential(
            nn.Linear(combined_dim, 64),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(64, 32),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(32, num_classes)
        )

    def forward(self, x):
        x = self.input_proj(x)

        lstm_out, (h_n, c_n) = self.lstm(x)
        lstm_out = self.layer_norm(lstm_out)

        if self.bidirectional:
            last_forward = h_n[-2]
            last_backward = h_n[-1]
            last_hidden = torch.cat([last_forward, last_backward], dim=1)
        else:
            last_hidden = h_n[-1]

        attn_scores = self.attention(lstm_out)
        attn_weights = torch.softmax(attn_scores, dim=1)
        attn_pooled = torch.sum(lstm_out * attn_weights, dim=1)

        mean_pooled = torch.mean(lstm_out, dim=1)
        max_pooled, _ = torch.max(lstm_out, dim=1)

        features = torch.cat(
            [last_hidden, attn_pooled, mean_pooled, max_pooled],
            dim=1
        )

        features = self.dropout(features)
        logits = self.classifier(features)
        return logits

# ==================== FOCAL LOSS ====================

class FocalLoss(nn.Module):
    def __init__(self, alpha=None, gamma=2.0, reduction='mean'):
        super().__init__()
        self.alpha = alpha
        self.gamma = gamma
        self.reduction = reduction

    def forward(self, inputs, targets):
        log_probs = F.log_softmax(inputs, dim=1)
        probs = torch.exp(log_probs)

        targets = targets.view(-1, 1)

        log_pt = log_probs.gather(1, targets).squeeze(1)
        pt = probs.gather(1, targets).squeeze(1)

        if self.alpha is not None:
            at = self.alpha.gather(0, targets.squeeze(1))
            loss = -at * ((1 - pt) ** self.gamma) * log_pt
        else:
            loss = -((1 - pt) ** self.gamma) * log_pt

        if self.reduction == 'mean':
            return loss.mean()
        elif self.reduction == 'sum':
            return loss.sum()
        else:
            return loss


In [9]:
model_configs = {
    "RNN": {
        "model_type": "rnn",
        "hidden_dim": 32,
        "num_layers": 1,
        "dropout": 0.2,
        "bidirectional": False,
    },
    "ImprovedLSTM": {
        "model_type": "improved_lstm",
        "hidden_dim": 64,
        "num_layers": 2,
        "dropout": 0.3,
        "bidirectional": True,
    },
    "LSTM": {
        "model_type": "lstm",
        "hidden_dim": 32,
        "num_layers": 2,
        "dropout": 0.3,
        "bidirectional": False,
    },
    "BiLSTM": {
        "model_type": "lstm",
        "hidden_dim": 32,
        "num_layers": 2,
        "dropout": 0.3,
        "bidirectional": True,
    },
    "GRU": {
        "model_type": "gru",
        "hidden_dim": 32,
        "num_layers": 2,
        "dropout": 0.3,
        "bidirectional": False,
    },
    "ImprovedGRU": {
        "model_type": "improved_gru",
        "hidden_dim": 64,
        "num_layers": 2,
        "dropout": 0.3,
        "bidirectional": True,
    },
}

In [10]:
def build_model(config, input_dim, num_classes=2):
    if config["model_type"] == "rnn":
        model = RNNClassifier(
            input_dim=input_dim,
            hidden_dim=config["hidden_dim"],
            num_layers=config["num_layers"],
            num_classes=num_classes,
            dropout=config["dropout"],
            bidirectional=config["bidirectional"]
        )

    elif config["model_type"] == "lstm":
        model = LSTMClassifier(
            input_dim=input_dim,
            hidden_dim=config["hidden_dim"],
            num_layers=config["num_layers"],
            num_classes=num_classes,
            dropout=config["dropout"],
            bidirectional=config["bidirectional"]
        )

    elif config["model_type"] == "gru":
        model = GRUBaseClassifier(
            input_dim=input_dim,
            hidden_dim=config["hidden_dim"],
            num_layers=config["num_layers"],
            num_classes=num_classes,
            dropout=config["dropout"],
            bidirectional=config["bidirectional"]
        )

    elif config["model_type"] == "improved_gru":
        model = ImprovedGRUClassifier(
            input_dim=input_dim,
            hidden_dim=config["hidden_dim"],
            num_layers=config["num_layers"],
            num_classes=num_classes,
            dropout=config["dropout"],
            bidirectional=config["bidirectional"]
        )
    elif config["model_type"] == "improved_lstm":
        model = ImprovedLSTMClassifier(
            input_dim=input_dim,
            hidden_dim=config["hidden_dim"],
            num_layers=config["num_layers"],
            num_classes=num_classes,
            dropout=config["dropout"],
            bidirectional=config["bidirectional"]
    )

    else:
        raise ValueError(f"Неизвестный тип модели: {config['model_type']}")

    return model.to(device)

In [11]:
for model_name, config in model_configs.items():
    model = build_model(config, input_dim=n_features)
    n_params = sum(p.numel() for p in model.parameters())
    logger.info(f"{model_name}: {n_params:,} параметров")

2026-04-05 16:18:00,573 - INFO - RNN: 3,586 параметров
2026-04-05 16:18:00,586 - INFO - ImprovedLSTM: 210,147 параметров
2026-04-05 16:18:00,592 - INFO - LSTM: 18,370 параметров
2026-04-05 16:18:00,601 - INFO - BiLSTM: 44,482 параметров
2026-04-05 16:18:00,606 - INFO - GRU: 14,146 параметров
2026-04-05 16:18:00,618 - INFO - ImprovedGRU: 168,675 параметров


## Часть 5: Обучение и сравнение моделей

In [12]:
def train_epoch(model, train_loader, optimizer, criterion, device, clip_grad=1.0):
    """Обучение на одной эпохе с mini-batches."""
    model.train()
    total_loss = 0.0

    for X_batch, y_batch in train_loader:
        X_batch = X_batch.to(device)
        y_batch = y_batch.to(device)

        optimizer.zero_grad()
        logits = model(X_batch)
        loss = criterion(logits, y_batch)
        loss.backward()

        if clip_grad > 0:
            nn.utils.clip_grad_norm_(model.parameters(), clip_grad)

        optimizer.step()
        total_loss += loss.item() * X_batch.size(0)

    return total_loss / len(train_loader.dataset)


def validate(model, val_loader, criterion, device):
    """Валидация модели."""
    model.eval()
    total_loss = 0.0
    all_probs = []
    all_labels = []

    with torch.no_grad():
        for X_batch, y_batch in val_loader:
            X_batch = X_batch.to(device)
            y_batch = y_batch.to(device)

            logits = model(X_batch)
            loss = criterion(logits, y_batch)
            total_loss += loss.item() * X_batch.size(0)

            probs = torch.softmax(logits, dim=1)[:, 1].cpu().numpy()
            labels = y_batch.cpu().numpy()

            all_probs.extend(probs)
            all_labels.extend(labels)

    all_probs = np.array(all_probs)
    all_labels = np.array(all_labels)
    pr_auc = average_precision_score(all_labels, all_probs)

    return total_loss / len(val_loader.dataset), pr_auc

In [13]:
def predict_on_loader(model, data_loader, device):
    """
    Получение вероятностей и истинных меток на произвольном loader.
    """
    model.eval()
    all_probs = []
    all_labels = []

    with torch.no_grad():
        for X_batch, y_batch in data_loader:
            X_batch = X_batch.to(device)
            y_batch = y_batch.to(device)

            logits = model(X_batch)
            probs = torch.softmax(logits, dim=1)[:, 1].cpu().numpy()

            all_probs.extend(probs)
            all_labels.extend(y_batch.cpu().numpy())

    y_score = np.array(all_probs)
    y_true = np.array(all_labels)

    return y_true, y_score


def find_best_threshold(y_true, y_score, beta=0.5):
    """
    Подбор оптимального порога по F-beta.
    """
    thresholds = np.arange(0.0, 1.01, 0.01)
    fbeta_scores = []

    for thresh in thresholds:
        y_pred_thresh = (y_score >= thresh).astype(int)
        score = fbeta_score(y_true, y_pred_thresh, beta=beta, zero_division=0)
        fbeta_scores.append(score)

    best_idx = int(np.argmax(fbeta_scores))
    best_threshold = thresholds[best_idx]
    best_score = fbeta_scores[best_idx]

    return best_threshold, best_score, thresholds, fbeta_scores


def calculate_metrics(y_true, y_score, threshold):
    """
    Расчет всех основных метрик качества.
    """
    y_pred = (y_score >= threshold).astype(int)

    tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()

    metrics = {
        'ROC-AUC': roc_auc_score(y_true, y_score),
        'PR-AUC': average_precision_score(y_true, y_score),
        'Accuracy': accuracy_score(y_true, y_pred),
        'Precision': precision_score(y_true, y_pred, zero_division=0),
        'Recall': recall_score(y_true, y_pred, zero_division=0),
        'F1': f1_score(y_true, y_pred, zero_division=0),
        'F0.5': fbeta_score(y_true, y_pred, beta=0.5, zero_division=0),
        'Specificity': tn / (tn + fp) if (tn + fp) > 0 else 0.0,
        'FPR': fp / (fp + tn) if (fp + tn) > 0 else 0.0,
        'FNR': fn / (fn + tp) if (fn + tp) > 0 else 0.0,
        'NPV': tn / (tn + fn) if (tn + fn) > 0 else 0.0,
        'TN': tn,
        'FP': fp,
        'FN': fn,
        'TP': tp,
    }

    return y_pred, metrics

## Часть 6: Сравнительная оценка моделей

In [15]:
results = []
trained_models = {}
training_histories = {}

common_batch_size = 64
common_num_epochs = 40
common_patience = 8

train_loader = DataLoader(
    TensorDataset(X_train_tensor, y_train_tensor),
    batch_size=common_batch_size,
    shuffle=True
)

val_loader = DataLoader(
    TensorDataset(X_val_tensor, y_val_tensor),
    batch_size=common_batch_size,
    shuffle=False
)

test_loader = DataLoader(
    TensorDataset(X_test_tensor, y_test_tensor),
    batch_size=common_batch_size,
    shuffle=False
)

for model_name, config in model_configs.items():
    reset_seeds(42)

    train_loader = DataLoader(
        TensorDataset(X_train_tensor, y_train_tensor),
        batch_size=common_batch_size,
        shuffle=True
    )

    val_loader = DataLoader(
        TensorDataset(X_val_tensor, y_val_tensor),
        batch_size=common_batch_size,
        shuffle=False
    )

    test_loader = DataLoader(
        TensorDataset(X_test_tensor, y_test_tensor),
        batch_size=common_batch_size,
        shuffle=False
    )
    logger.info(f"\n{'=' * 90}")
    logger.info(f"Запуск модели: {model_name}")
    logger.info(f"Конфиг: {config}")
    logger.info(f"{'=' * 90}")

    model = build_model(config, input_dim=n_features)

    n_params = sum(p.numel() for p in model.parameters())

    criterion = nn.CrossEntropyLoss(
        weight=torch.tensor([1.0, 10.0], dtype=torch.float32).to(device)
    )

    optimizer = optim.AdamW(
        model.parameters(),
        lr=1e-3,
        weight_decay=1e-5
    )

    scheduler = optim.lr_scheduler.StepLR(
        optimizer,
        step_size=10,
        gamma=0.5
    )

    best_pr_auc = 0.0
    patience_counter = 0
    best_state_dict = None

    train_losses = []
    val_losses = []
    val_pr_aucs = []
    learning_rates = []

    for epoch in range(common_num_epochs):
        train_loss = train_epoch(model, train_loader, optimizer, criterion, device)
        val_loss, val_pr_auc = validate(model, val_loader, criterion, device)

        scheduler.step()

        train_losses.append(train_loss)
        val_losses.append(val_loss)
        val_pr_aucs.append(val_pr_auc)
        learning_rates.append(optimizer.param_groups[0]['lr'])

        if (epoch + 1) % 5 == 0:
            logger.info(
                f"{model_name} | эпоха {epoch + 1:02d} | "
                f"train_loss={train_loss:.4f} | "
                f"val_loss={val_loss:.4f} | "
                f"val_pr_auc={val_pr_auc:.4f}"
            )

        if val_pr_auc > best_pr_auc:
            best_pr_auc = val_pr_auc
            patience_counter = 0
            best_state_dict = {k: v.cpu().clone() for k, v in model.state_dict().items()}
        else:
            patience_counter += 1
            if patience_counter >= common_patience:
                logger.info(f"{model_name}: ранняя остановка на эпохе {epoch + 1}")
                break

    if best_state_dict is not None:
        model.load_state_dict(best_state_dict)
        model.to(device)

    y_val_true, y_val_score = predict_on_loader(model, val_loader, device)
    best_threshold, best_f05, thresholds, fbeta_scores = find_best_threshold(
        y_val_true,
        y_val_score,
        beta=0.5
    )

    y_test_true, y_test_score = predict_on_loader(model, test_loader, device)
    y_test_pred, test_metrics = calculate_metrics(
        y_test_true,
        y_test_score,
        threshold=best_threshold
    )

    result_row = {
        "Model": model_name,
        "Params": n_params,
        "Best Val PR-AUC": best_pr_auc,
        "Best Threshold": best_threshold,
        **test_metrics
    }
    results.append(result_row)

    trained_models[model_name] = model
    training_histories[model_name] = {
        "train_losses": train_losses,
        "val_losses": val_losses,
        "val_pr_aucs": val_pr_aucs,
        "learning_rates": learning_rates,
        "y_val_true": y_val_true,
        "y_val_score": y_val_score,
        "y_test_true": y_test_true,
        "y_test_score": y_test_score,
        "y_test_pred": y_test_pred,
        "best_threshold": best_threshold,
        "best_f05": best_f05,
    }

    logger.info(
        f"{model_name}: test PR-AUC={test_metrics['PR-AUC']:.4f}, "
        f"ROC-AUC={test_metrics['ROC-AUC']:.4f}, "
        f"F0.5={test_metrics['F0.5']:.4f}, "
        f"Precision={test_metrics['Precision']:.4f}, "
        f"Recall={test_metrics['Recall']:.4f}"
    )

results_df = pd.DataFrame(results)
results_df = results_df.sort_values(by=["PR-AUC", "F0.5"], ascending=False).reset_index(drop=True)

print("\nСводная таблица результатов:")
display(results_df)

2026-04-05 16:22:40,875 - INFO - 
2026-04-05 16:22:40,876 - INFO - Запуск модели: RNN
2026-04-05 16:22:40,877 - INFO - Конфиг: {'model_type': 'rnn', 'hidden_dim': 32, 'num_layers': 1, 'dropout': 0.2, 'bidirectional': False}
2026-04-05 16:22:40,877 - INFO - ==========================================================================================
2026-04-05 16:22:44,754 - INFO - RNN | эпоха 05 | train_loss=0.2850 | val_loss=0.2522 | val_pr_auc=0.5359
2026-04-05 16:22:48,991 - INFO - RNN | эпоха 10 | train_loss=0.1666 | val_loss=0.1031 | val_pr_auc=0.7674
2026-04-05 16:22:52,918 - INFO - RNN | эпоха 15 | train_loss=0.1338 | val_loss=0.1042 | val_pr_auc=0.7440
2026-04-05 16:22:56,701 - INFO - RNN | эпоха 20 | train_loss=0.1031 | val_loss=0.1273 | val_pr_auc=0.6913
2026-04-05 16:22:56,701 - INFO - RNN: ранняя остановка на эпохе 20
2026-04-05 16:22:56,918 - INFO - RNN: test PR-AUC=0.7910, ROC-AUC=0.9974, F0.5=0.6154, Precision=0.5714, Recall=0.8889
2026-04-05 16:22:56,920 - INFO - 
2026-04-


Сводная таблица результатов:


,Model,Params,Best Val PR-AUC,Best Threshold,ROC-AUC,PR-AUC,Accuracy,Precision,Recall,F1,F0.5,Specificity,FPR,FNR,NPV,TN,FP,FN,TP
0,GRU,14146,0.889908,0.96,0.998198,0.867784,0.994660,0.727273,0.888889,0.800000,0.754717,0.995946,0.004054,0.111111,0.998645,737,3,1,8
1,RNN,3586,0.772203,0.72,0.997447,0.791037,0.990654,0.571429,0.888889,0.695652,0.615385,0.991892,0.008108,0.111111,0.998639,734,6,1,8
2,ImprovedGRU,168675,0.908096,0.96,0.992042,0.717382,0.991989,0.800000,0.444444,0.571429,0.689655,0.998649,0.001351,0.555556,0.993280,739,1,5,4
3,ImprovedLSTM,210147,0.869250,0.93,0.990240,0.713215,0.989319,0.571429,0.444444,0.500000,0.540541,0.995946,0.004054,0.555556,0.993261,737,3,5,4
4,BiLSTM,44482,0.887428,0.82,0.997447,0.700297,0.995995,0.750000,1.000000,0.857143,0.789474,0.995946,0.004054,0.000000,1.000000,737,3,0,9
5,LSTM,18370,0.872680,0.93,0.995495,0.636541,0.989319,0.555556,0.555556,0.555556,0.555556,0.994595,0.005405,0.444444,0.994595,736,4,4,5
